# Stage 1 / Step 1B - Component 1 via sentence-transformer embeddings (Option B)

Frozen transformer (`all-MiniLM-L6-v2`) turns the post text into a semantic vector; a light
LogisticRegression head predicts P(viral). The embedder is NOT trained -> only the cheap head is
refit as data grows, and it handles new vocabulary/trends (the weak point of TF-IDF in a
continuously-collected real-time stream).

Long transcripts exceed the model's token limit, so each document is split into word-chunks,
every chunk embedded, then mean-pooled into one document vector. Compared against the TF-IDF
baseline (Option A, PR-AUC 0.604).

In [ ]:
# load data + label + frozen embedder
from pathlib import Path
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import average_precision_score, roc_auc_score
import joblib

ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_parquet(ROOT / "ml" / "data" / "video_features.parquet")

thr = df["virality_score"].quantile(0.75)
df["is_viral"] = (df["virality_score"] >= thr).astype(int)
y = df["is_viral"]

embedder = SentenceTransformer("all-MiniLM-L6-v2")   # frozen, CPU
DIM = embedder.get_sentence_embedding_dimension()
print("rows:", len(df), "| embedding dim:", DIM)

In [ ]:
# chunk each document, embed all chunks in one batch, mean-pool back to one vector per document
WORDS_PER_CHUNK, MAX_CHUNKS = 180, 12

all_chunks, slices = [], []
for text in df["text_all"].fillna(""):
    words = text.split()
    chunks = [" ".join(words[i:i + WORDS_PER_CHUNK])
              for i in range(0, len(words), WORDS_PER_CHUNK)][:MAX_CHUNKS] or [""]
    start = len(all_chunks)
    all_chunks.extend(chunks)
    slices.append((start, len(all_chunks)))

print(f"documents: {len(slices)} | total chunks: {len(all_chunks)}")
emb_all = embedder.encode(all_chunks, batch_size=64, convert_to_numpy=True,
                          normalize_embeddings=True, show_progress_bar=True)
X_emb = np.vstack([emb_all[s:e].mean(axis=0) for s, e in slices])
print("document-embedding matrix:", X_emb.shape)

In [ ]:
# light head on embeddings -> OOF P(viral), compare to Option A (0.604)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
head = LogisticRegression(max_iter=2000, class_weight="balanced")

oof = cross_val_predict(head, X_emb, y, cv=cv, method="predict_proba")[:, 1]
ap = average_precision_score(y, oof)
auc = roc_auc_score(y, oof)
print(f"Option B (embeddings)  PR-AUC: {ap:.3f}  |  ROC-AUC: {auc:.3f}")
print(f"Option A (TF-IDF)      PR-AUC: 0.604  |  ROC-AUC: 0.830  (reference)")

In [ ]:
# fit final head on all data, save head + embedder name + OOF scores for the fusion step
head.fit(X_emb, y)
df["content_score"] = oof

out_dir = ROOT / "ml" / "models"
out_dir.mkdir(parents=True, exist_ok=True)
joblib.dump({"embedder_name": "all-MiniLM-L6-v2",
             "words_per_chunk": WORDS_PER_CHUNK, "max_chunks": MAX_CHUNKS,
             "head": head}, out_dir / "stage1_content_embed.joblib")

df[["video_id", "is_viral", "content_score"]].to_parquet(
    ROOT / "ml" / "data" / "stage1_scores.parquet", index=False)
print("Saved embedder+head ->", out_dir / "stage1_content_embed.joblib")
print("Saved scores -> ml/data/stage1_scores.parquet")